# w2 · Per-cycle sequence preprocessing (as_of = expiry − k)

`w2_sequence_preprocessing.ipynb` ends every customer's 90-day window at the cohort cutoff (2017-01-31 for February, 2017-02-28 for March). The renewal decision, however, happens at each customer's **expiry**, and `eda/10_per_cycle_prediction.ipynb` and section 7 of `eda/07_trajectories.ipynb` showed that scoring each customer **k days before their own expiry** is more informative. This notebook builds that tensor for w2.

| | Calendar export (`export-sequential-v6-manual`) | This export (`export-sequential-v6-cycle-k7-manual`) |
|---|---|---|
| Window end | cohort cutoff, same for everyone | `as_of = anchor − k`, per customer |
| Customers | manual at the cutoff | manual at the reference date, expiry in the label month, eligible at `as_of` |
| Channels | 14 | the same 14, built by the same code |

**Definitions** (as in notebook 10 of the EDA):

- **reference date**: 2017-01-31 (February cohort) and 2017-02-28 (March cohort);
- **anchor**: the expiry known at the reference date, read from the last transaction day exactly as the tensor's state channels read it;
- **in scope**: the anchor falls inside the label month (1–28 days after 2017-01-31, 1–31 days after 2017-02-28). These are the customers the label is defined for;
- **segment**: manual at the reference date (last transaction is a payment without auto-renew), the same rule as the calendar manual run;
- **as_of** = anchor − k. Every channel uses only records dated on or before `as_of`; nothing from the expiry day onwards is used;
- **eligibility (no look-ahead)**: when `as_of` is before the reference date, the anchor itself comes from the future, so the customer is kept only if the expiry already known at `as_of` equals the anchor. When `as_of` is on or after the reference date, everyone is kept.

**How the tensor is built.** The existing builders take a single cutoff. Instead of changing them, each customer's dates (transactions, expiries, listening days) are shifted by `reference − as_of` days, so that their `as_of` falls on the reference date, and the unchanged builder runs with that cutoff. The channels depend only on differences between dates, so the result is the window ending at `as_of`. Section 4 checks this against a direct build with `cutoff = as_of`.

The labels are the official ones (`train` for February, `train_v2` for March), unchanged.

**Channels.** The same 14 channels as the calendar export (`EXPORT_CHANNELS` in `src/data_processing/constants.py`): the 8 listening channels, the payment events `tx_count`, `tx_paid_sum`, `tx_plan_days_sum` and the payment states `tx_state_known`, `tx_days_since_last`, `tx_days_to_expire`. The builder computes its 20 internal channels (the audit and the checks use them) and the export keeps these 14.

In [1]:
pip install polars

Defaulting to user installation because normal site-packages is not writeable


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [2]:
import json
import os
import resource
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

sys.path.insert(0, str(Path.cwd().parent))   # make the `w2` package importable

from w2.src.data_processing.constants import LSTM_CHANNELS, SEQ_DAYS
from w2.src.data_processing.users import load_target_users, users_with_codes
from w2.src.data_processing.transactions import scan_transactions
from w2.src.data_processing.user_logs import build_user_day_logs
from w2.src.data_processing.sequences.cohort_export import write_cohort_sequences
from w2.src.data_processing.sequences.dataset_export import export_sequence_dataset, verify_export, export_summary
from w2.src.data_processing.sequences.cycle_export import (
    build_labels, check_shift_equivalence, cycle_export_name, cycle_population,
    last_day_days_to_expire, sequence_users, shift_to_reference, window_bounds)

K = int(os.environ.get("W2_CYCLE_K", "7"))          # days before the expiry
SEGMENT = "manual"                                   # w2 uses only the manual segment
REFERENCE = {"train_feb": "2017-01-31", "test": "2017-02-28"}   # also the `cutoffs` of the export contract
LABEL_MONTHS = {"train_feb": "2017-02", "test": "2017-03"}
LABEL_FILES = {"train_feb": "train", "test": "train_v2"}
MAX_DAYS = {"train_feb": 28, "test": 31}                       # length of the label month
SPLITS = list(LABEL_FILES)
from w2.src.data_processing.constants import DATASET_VERSION, EXPORT_CHANNELS   # contract v6: 14 channels

PARQUET_DIR = Path("../data/parquet")
EXPORT_DIR = Path(cycle_export_name(K, SEGMENT))
print("k =", K, "| segment:", SEGMENT, "| export:", EXPORT_DIR)

k = 7 | segment: manual | export: export-sequential-v6-cycle-k7-manual


## 2. Population: anchor, segment and eligibility

In [3]:
labelled_users, user_index = load_target_users(PARQUET_DIR, LABEL_FILES)
users_by_split = {s: users_with_codes(labelled_users[s], user_index) for s in SPLITS}

# Transactions up to the last possible as_of (the last day of the March label month minus 1).
transactions = scan_transactions(PARQUET_DIR, user_index, "2017-03-30").collect(engine="streaming")
print("transactions:", transactions.shape)

population, reports = {}, []
for split in SPLITS:
    population[split], report = cycle_population(
        transactions, users_by_split[split], labelled_users[split], REFERENCE[split], MAX_DAYS[split], K, SEGMENT)
    reports += [{"cohort": split, **row} for row in report]
pd.DataFrame(reports)

transactions: (17439752, 10)


,cohort,step,customers,churn_rate
0,train_feb,cohort,992931,0.063923
1,train_feb,segment = manual at 2017-01-31,111751,0.321509
2,train_feb,anchor 1-28 days after 2017-01-31,104970,0.293103
3,train_feb,eligible at as_of = anchor - 7,104964,0.293091
4,train_feb,dropped (anchor not yet known at as_of),6,0.500000
5,test,cohort,970960,0.089942
6,test,segment = manual at 2017-02-28,113870,0.410995
7,test,anchor 1-31 days after 2017-02-28,104465,0.383794
8,test,eligible at as_of = anchor - 7,104322,0.383131
9,test,dropped (anchor not yet known at as_of),143,0.867133


The eligibility filter removes almost nobody at k = 7: the anchor is the expiry of a payment that was already made, so it is known well before the expiry. Note the March churn rate of the manual segment is higher than February's for reasons that are about the label, not the customers (`eda/12_march_shift_diagnosis.ipynb`).

In [4]:
rows = []
for split in SPLITS:
    p = population[split]
    rows.append({"cohort": split, "customers": p.height, "churn_rate": p["is_churn"].mean(),
                 "first as_of": p["as_of"].min(), "last as_of": p["as_of"].max(),
                 "as_of after the reference date": (p["as_of"] > date.fromisoformat(REFERENCE[split])).mean(),
                 "shift (days) min / max": f"{p['offset'].min()} / {p['offset'].max()}"})
pd.DataFrame(rows)

,cohort,customers,churn_rate,first as_of,last as_of,as_of after the reference date,shift (days) min / max
0,train_feb,104964,0.293091,2017-01-25,2017-02-21,0.737234,-21 / 6
1,test,104322,0.383131,2017-02-22,2017-03-24,0.801480,-24 / 6


## 3. Listening logs, date shift and the 14-channel tensor

One user-day table covers every window of both cohorts. Each cohort's transactions and logs are then shifted with that cohort's offsets (a customer present in both months is shifted twice, independently), and `write_cohort_sequences`, the same function as the calendar export, builds the tensor with the reference date as cutoff.

In [5]:
pop_index = user_index.join(pl.concat([population[s].select("uid") for s in SPLITS]).unique(), on="uid", how="semi")
bounds = [window_bounds(population[s], SEQ_DAYS) for s in SPLITS]
LOG_START, LOG_END = min(b[0] for b in bounds), max(b[1] for b in bounds)
user_day = build_user_day_logs(PARQUET_DIR, pop_index, LOG_START, LOG_END)
print("user-day rows:", f"{user_day.height:,}", "| from", LOG_START, "to", LOG_END)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / "metadata.json").unlink(missing_ok=True)
lstm, audit, users, labels, shifted = {}, {}, {}, {}, {}
for split in SPLITS:
    users[split] = sequence_users(population[split])
    labels[split] = build_labels(population[split])
    txn_s, logs_s = shift_to_reference(transactions, user_day, population[split])
    lstm[split], audit[split] = write_cohort_sequences(logs_s, txn_s, users[split], REFERENCE[split], EXPORT_DIR, split)
    print(split, "tensor:", lstm[split].shape, "| customers without logs:",
          audit[split]["listening"]["users_without_logs"])
_ = (EXPORT_DIR / "sequence_audit.json").write_text(json.dumps(audit, indent=2, default=str))

user-day rows: 12,617,629 | from 2016-10-28 to 2017-03-24


train_feb tensor: (104964, 90, 14) | customers without logs: 732


test tensor: (104322, 90, 14) | customers without logs: 420


## 4. Checks

1. **Shift equivalence.** For the three most frequent `as_of` dates of each cohort, up to 300 customers are rebuilt **without any shift**, with the unchanged builders and `cutoff = as_of`. Their rows must be identical to the exported ones.
2. **Last day = k days before the anchor.** On the last day of the window, `tx_days_to_expire` must equal k for every customer whose expiry known at `as_of` is the anchor. It can differ only for customers who transacted between the reference date and `as_of` (an early payment or a cancellation), which is information the model is allowed to see.

In [6]:
checks = []
for split in SPLITS:
    for row in check_shift_equivalence(transactions, user_day, population[split], REFERENCE[split], lstm[split]):
        checks.append({"cohort": split, **row})
pd.DataFrame(checks)

,cohort,as_of,customers,identical
0,train_feb,2017-01-31,300,True
1,train_feb,2017-01-30,300,True
2,train_feb,2017-01-29,300,True
3,test,2017-03-14,300,True
4,test,2017-03-07,300,True
5,test,2017-03-12,300,True


In [7]:
dte_channel = EXPORT_CHANNELS.index("tx_days_to_expire")
rows = []
for split in SPLITS:
    dte = last_day_days_to_expire(lstm[split], dte_channel)
    p = population[split].with_columns(last_day_dte=pl.Series(dte))
    after = pl.col("as_of") > pl.lit(date.fromisoformat(REFERENCE[split]))
    for name, sub in (("as_of on or before the reference date", p.filter(~after)), ("as_of after the reference date", p.filter(after))):
        rows.append({"cohort": split, "group": name, "customers": sub.height,
                     f"days_to_expire == {K} on the last day": (sub["last_day_dte"] == K).mean(),
                     "churn when == k": sub.filter(pl.col("last_day_dte") == K)["is_churn"].mean(),
                     "churn when != k": sub.filter(pl.col("last_day_dte") != K)["is_churn"].mean()})
    assert (p.filter(~after)["last_day_dte"] == K).all(), "a customer with as_of <= reference does not end k days before the anchor"
pd.DataFrame(rows)

,cohort,group,customers,days_to_expire == 7 on the last day,churn when == k,churn when != k
0,train_feb,as_of on or before the reference date,27581,1.000000,0.271745,NaN
1,train_feb,as_of after the reference date,77383,0.977398,0.292025,0.675815
2,test,as_of on or before the reference date,20710,1.000000,0.384983,NaN
3,test,as_of after the reference date,83612,0.979154,0.377102,0.644291


For customers scored before the reference date, every window ends exactly k days before the anchor. For those scored after it, a minority already transacted again before `as_of` (most often a payment made before the expiry). Notebook 11 of the EDA showed that the label does not count such a payment as a renewal, so these customers churn much more often; the model sees that event in the transaction channels.

## 5. Export

In [8]:
run_config = {
    "cutoffs": REFERENCE, "label_months": LABEL_MONTHS, "label_files": LABEL_FILES,
    "splits": SPLITS, "seq_days": SEQ_DAYS,
    "log_sources": ["user_logs.parquet", "user_logs_v2.parquet"],
    "alignment": {
        "type": "cycle", "k": K, "segment": SEGMENT,
        "as_of": "anchor - k, per customer; the 90-day window ends on as_of",
        "anchor": "expiry of the last transaction day known at the reference date (`cutoffs`)",
        "scope": "anchor 1-28 days (February) / 1-31 days (March) after the reference date",
        "eligibility": "as_of >= reference date, or expiry known at as_of == anchor",
        "construction": "dates shifted by reference - as_of, calendar builder with cutoff = reference",
    },
    "windows": {s: {"first_as_of": str(population[s]["as_of"].min()), "last_as_of": str(population[s]["as_of"].max())}
                for s in SPLITS},
    "listening_encoding": "log1p, unscaled (scalers are fit in the model notebook)",
    "sample_users": None,
}
metadata = export_sequence_dataset(EXPORT_DIR, lstm, labels, users, audit, run_config, dataset_version=DATASET_VERSION,
                                   channels=EXPORT_CHANNELS)
# Per-row anchor and as_of, to compare with other runs on the same customers.
for split in SPLITS:
    population[split].with_row_index("row").select("row", "msno", "anchor", "as_of", "offset") \
        .write_parquet(EXPORT_DIR / f"cycle_index_{split}.parquet")
verify_export(EXPORT_DIR, users)
print("Export verified:", EXPORT_DIR.resolve())
print("Peak RAM:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")
export_summary(EXPORT_DIR)

Export verified: /home/alejandropenagos/Escritorio/Alejandro/Machine Learning/customer-churn-prediction/workshops/w2/export-sequential-v6-cycle-k7-manual
Peak RAM: 4.59 GB


,file,MB
0,cycle_index_test.parquet,3.93
1,cycle_index_train_feb.parquet,3.95
2,labels_test.parquet,3.74
3,labels_train_feb.parquet,3.76
4,metadata.json,0.01
5,seq_lstm_test.npy,262.89
6,seq_lstm_train_feb.npy,264.51
7,sequence_audit.json,0.01


## 6. What the model side needs

The tensor has the same 14-channel contract (v6) as `export-sequential-v6-manual`, so the models and the scalers work unchanged. Only the run selection changes: `w2_neural_models_cycle.ipynb` sets `W2_ALIGNMENT=cycle` and `W2_CYCLE_K` before importing `src.models`, which makes it read this export and write to `artifacts/sherlock_w2_v6_manual_cycle_k7/`. The feature preparation does not filter rows by segment again, because the population was already fixed here at the reference date.